In [1]:
from datetime import datetime
from pathlib import Path
import joblib
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rcParams
import functools
# from logger.logger import get_logger, setup_logging
import optuna
from logger.logger import get_logger, setup_logging
from model.Granger_causalFormer import PredictModel
from data_loader import TimeSeriesDataloader

# from util import read_json
from train_causalformer import CausalFormerTrainer

In [2]:

# 参数设置
# P = 5           # 时间序列的数量
# T = 1000        # 总时间点
LAG = 2         # 真实的 VAR 滞后
SPARSITY = 0.4  # 格兰杰因果矩阵的稀疏度
BETA_VALUE = 0.8# 系数值
SD = 0.1        # 噪声的标准差
DATA_SEED = 42  # 用于可重复性的随机种子
INPUT_WINDOW = 20 # 输入序列长度 (输入窗口)
FEATURE_DIM = 1 # 每个时间序列在每个时间点上的特征数量
OUTPUT_DIM = 1  # 每个时间序列在每个预测时间步上输出的目标数量
OUTPUT_WINDOW = 1 # 预测下一个时间步


# --- 训练和 Optuna 参数 ---
EPOCHS = 10
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_TRIALS = 50 # Optuna 的试验次数
STUDY_NAME = "optuna"


# --- 1. 生成并预处理数据 ---
# X_np, _, GC_true_np = simulate_var(p=P, T=T, lag=LAG, sparsity=SPARSITY,
#                                    beta_value=BETA_VALUE, sd=SD, seed=DATA_SEED)

# --- 1. 得到数据并预处理 ---
data_path = 'data/fMRI/timeseries9.csv'
true_gc_path = 'data/fMRI/sim9_gt_processed.csv'

timeseriesDataLoader = TimeSeriesDataloader(data_dir=data_path, gc_dir=true_gc_path, batch_size=BATCH_SIZE, 
                                            DATA_SEED=DATA_SEED, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW)
GC_true_np = timeseriesDataLoader.get_true_granger() # 得到真实的格兰杰因果矩阵
# train_loader, val_loader, test_loader = timeseriesDataLoader.split_sampler() # 得到训练集、验证集和测试集的数据加载器

In [9]:
X_train_seq, y_train_seq = timeseriesDataLoader.split_sampler() # 得到训练集、验证集和测试集的数据加载器
# print("train_loader:", X_train_seq)
print("val_loader:", y_train_seq)

val_loader: [[[[ 1.7637  ]
   [-2.6928  ]
   [-1.9959  ]
   [ 0.25264 ]
   [ 0.94108 ]]]


 [[[-3.4837  ]
   [-0.060278]
   [-4.9526  ]
   [-1.4     ]
   [-4.0084  ]]]


 [[[-7.8677  ]
   [-1.421   ]
   [-2.6271  ]
   [-0.66188 ]
   [-0.42175 ]]]


 ...


 [[[-3.5075  ]
   [-4.8436  ]
   [-0.33372 ]
   [-2.6817  ]
   [-8.5151  ]]]


 [[[-0.81555 ]
   [-3.4397  ]
   [-0.85458 ]
   [-2.2062  ]
   [-3.5635  ]]]


 [[[-1.8214  ]
   [-4.6542  ]
   [-3.6704  ]
   [-3.3715  ]
   [-1.1688  ]]]]


In [13]:
input_window = INPUT_WINDOW # 输入序列长度
output_window = OUTPUT_WINDOW # 固定预测下一步
d_model = 32      # QK 嵌入维度
n_head = 4         # 注意力头数
n_layers = 3                     # Encoder 层数
ffn_hidden = 64# FFN 隐藏层维度
dropout = 0.1               # Dropout
tau = 1               # Softmax 温度


# GrangerTCN 参数
tcn_layers = 2                 # TCN 块数
tcn_channels = 32 # TCN 通道数
tcn_kernel_size = 3 # TCN 核大小
tcn_dropout = 0          # TCN Dropout
# tcn_channel_list = [tcn_channels] * tcn_layers

# 近端梯度下降和稀疏性参数
criterion =  nn.MSELoss()
lr = 1e-4     # 学习率
lambda_reg = 1e-5
penalty_type = 'GL'
alpha_gsgl = 0.5 # 仅在GSGL的情况下才有意义，负责控制GSGL内部组稀疏和组内稀疏的平衡

In [14]:
config = {
    'data_loader': {
        'args': {
            'input_window': INPUT_WINDOW,
            'output_window': OUTPUT_WINDOW,
            'feature_dim': FEATURE_DIM,
            'output_dim': OUTPUT_DIM,
            'series_num': 5
        }
    },
    'device': DEVICE.type # 传递设备类型
}

model = PredictModel(config=config,
                        d_model=d_model,
                        n_head=n_head,
                        n_layers=n_layers,
                        tcn_channels=tcn_channels,
                        tcn_kernel_size=tcn_kernel_size,
                        tcn_dropout=tcn_dropout,
                        ffn_hidden=ffn_hidden,
                        drop_prob=dropout,
                        tau=tau).to(DEVICE)

In [15]:
first_layer_param = model
print(first_layer_param)
'''
model.encoder.layers[0]：访问模型的第一层编码器层。
.attention：访问该编码器层中的注意力机制。
.tcn_processor：访问注意力机制中的时间卷积网络（TCN）处理器。
.network_layers[0]：访问TCN处理器中的第一个时间块（TemporalBlock）。
.conv1：访问该时间块中的第一个卷积层。
.weight：获取该卷积层的权重参数。
'''


PredictModel(
  (encoder): Encoder(
    (emb): Embedding(
      (feature_emb): Linear(in_features=20, out_features=32, bias=True)
      (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (drop_out): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attention): MultiHeadAttention(
          (Wq): Linear(in_features=32, out_features=32, bias=True)
          (Wk): Linear(in_features=32, out_features=32, bias=True)
          (tcn_processors): ModuleList(
            (0-4): 5 x GrangerTCN(
              (network_layers): ModuleList(
                (0): TemporalBlock(
                  (conv1): Conv1d(4, 32, kernel_size=(3,), stride=(1,), padding=(2,))
                  (chomp1): Chomp1d()
                  (relu1): ReLU()
                  (dropout1): Dropout(p=0, inplace=False)
                  (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
                  (chomp2): Chomp1d()
                  (re

'\nmodel.encoder.layers[0]：访问模型的第一层编码器层。\n.attention：访问该编码器层中的注意力机制。\n.tcn_processor：访问注意力机制中的时间卷积网络（TCN）处理器。\n.network_layers[0]：访问TCN处理器中的第一个时间块（TemporalBlock）。\n.conv1：访问该时间块中的第一个卷积层。\n.weight：获取该卷积层的权重参数。\n'

In [10]:
def load_model(config, best_params, device):
    """
    根据最佳参数加载模型
    """
    # 从最佳参数中提取模型参数
    d_model = best_params['d_model']
    n_head = best_params['n_head']
    n_layers = best_params['n_layers']
    ffn_hidden = best_params['ffn_hidden']
    dropout = best_params['dropout']
    tau = best_params['tau']
    
    # GrangerTCN 参数
    tcn_layers = best_params['tcn_layers']
    tcn_channels = best_params['tcn_channels']
    tcn_kernel_size = best_params['tcn_kernel_size']
    tcn_dropout = best_params['tcn_dropout']
    tcn_channel_list = [tcn_channels] * tcn_layers
    
    # 创建并返回模型
    model = PredictModel(
        config=config,
        d_model=d_model,
        n_head=n_head,
        n_layers=n_layers,
        tcn_channels=tcn_channel_list,
        tcn_kernel_size=tcn_kernel_size,
        tcn_dropout=tcn_dropout,
        ffn_hidden=ffn_hidden,
        drop_prob=dropout,
        tau=tau
    ).to(device)
    
    return model
